In [30]:
import pandas as pd

In [31]:
df = pd.read_csv(r"\pypipeline\data\raw\static\hourly\aqi\limited\us_paro_hourly.csv")

In [32]:
df[df["value"]<0]["value"].count()

np.int64(3980)

### Feature Extraction


In [33]:
df_pm25 = df[df["parameter"]=="pm25"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"pm25","local":"date"}).reset_index(drop=True)

df_o3 = df[df["parameter"]=="o3"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"o3","local":"date"}).reset_index(drop=True)

In [34]:
df_merged = df_pm25.merge(df_o3, on="date", how="outer").sort_values("date").reset_index(drop=True)

### Remove negative values

In [35]:
df_merged.loc[df_merged["pm25"]<0, "pm25"]=pd.NA
df_merged.loc[df_merged["o3"]<0, "o3"]=pd.NA

In [36]:
df_merged["date"] = pd.to_datetime(df_merged["date"])
df_merged.set_index("date", inplace=True)

In [37]:
df_merged

,pm25,o3
date,,
2017-03-03 05:00:00+05:45,106.1,0.002
2017-03-03 06:00:00+05:45,134.5,0.002
2017-03-03 07:00:00+05:45,154.7,0.002
2017-03-03 08:00:00+05:45,155.4,0.003
2017-03-03 09:00:00+05:45,178.7,0.005
...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039
2021-03-12 21:00:00+05:45,58.0,0.030
2021-03-12 22:00:00+05:45,60.0,0.030


In [38]:
df_merged.sort_index

<bound method DataFrame.sort_index of                             pm25     o3
date                                   
2017-03-03 05:00:00+05:45  106.1  0.002
2017-03-03 06:00:00+05:45  134.5  0.002
2017-03-03 07:00:00+05:45  154.7  0.002
2017-03-03 08:00:00+05:45  155.4  0.003
2017-03-03 09:00:00+05:45  178.7  0.005
...                          ...    ...
2021-03-12 20:00:00+05:45   58.0  0.039
2021-03-12 21:00:00+05:45   58.0  0.030
2021-03-12 22:00:00+05:45   60.0  0.030
2021-03-12 23:00:00+05:45   69.0  0.030
2021-03-13 00:00:00+05:45   69.0  0.051

[32364 rows x 2 columns]>

### time delta check
from the result we can see most of the data is hourly but some have larger deltas, we need to force everything to hourly

In [39]:
deltas = df_merged.index.sort_values().diff().value_counts()
deltas.head()

date
0 days 01:00:00    31634
0 days 02:00:00      515
0 days 03:00:00      109
0 days 04:00:00       44
0 days 05:00:00       19
Name: count, dtype: int64

In [40]:
df_merged= df_merged.asfreq('H') # force to hourly frequency

C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_16400\1743174348.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_merged= df_merged.asfreq('H') # force to hourly frequency


In [ ]:
df_merged["hour"] = df_merged.index.hour
df_featured_enginnered["day_of_week"] = df_featured_enginnered.index.dayofweek
df_featured_enginnered["is_weekend"] = df_featured_enginnered["day_of_week"].isin([5,6]).astype(int)
df_featured_enginnered["is_night"]=df_featured_enginnered["hour"].isin([0,1,2,3,4,5]).astype(int)

### Finding gaps in time series data
#### Rules used
The gap in this case is the gap in the sensor data not the gap on time
- Small Gap: <= 6 Hours
- Medium Gap: > 6 Hours and <= 12
- Large Gap: > 12 Hours
- very large Gap >24 Hours
#### Gap Imputation Methods

##### Small Gap
use linear interpolation, or Kalman filter
##### Medium Gap
use interpolation + flags
##### Large Gap
donot impute, just leave it as it is
##### Very large gap
use segmentation methods so that data doesn't leak for lag features


In [41]:
df_merged

,pm25,o3
date,,
2017-03-03 05:00:00+05:45,106.1,0.002
2017-03-03 06:00:00+05:45,134.5,0.002
2017-03-03 07:00:00+05:45,154.7,0.002
2017-03-03 08:00:00+05:45,155.4,0.003
2017-03-03 09:00:00+05:45,178.7,0.005
...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039
2021-03-12 21:00:00+05:45,58.0,0.030
2021-03-12 22:00:00+05:45,60.0,0.030


In [42]:
df_merged["pm25_missing"] = df_merged["pm25"].isna().astype(int)
df_merged["o3_missing"] = df_merged["o3"].isna().astype(int) 

In [43]:
pm25_run_id = (df_merged["pm25_missing"] != df_merged["pm25_missing"].shift()).cumsum()
o3_run_id = (df_merged["o3_missing"] != df_merged["o3_missing"].shift()).cumsum()

In [44]:
pm25_gap_length = (df_merged["pm25_missing"].groupby(pm25_run_id).transform("sum"))
o3_gap_length = (df_merged["o3_missing"].groupby(o3_run_id).transform("sum"))

In [45]:
df_merged["pm25_gap_length"] = pm25_gap_length
df_merged["o3_gap_length"] = o3_gap_length

In [46]:
SMALL_GAP_THRESHOLD = 6
MEDIUM_GAP_THRESHOLD = 12
LARGE_GAP_THRESHOLD = 24   

#### Small Gap case (<=6 hours)
Used time based interpolation from pandas

In [51]:
df_imputation = df_merged.copy()

In [ ]:
small_gap_mask_pm25 = df_merged["pm25_gap_length"] <= SMALL_GAP_THRESHOLD
small_gap_mask_o3 = df_merged["o3_gap_length"] <= SMALL_GAP_THRESHOLD

df_imputation.loc[small_gap_mask_pm25, "pm25"] = df_imputation["pm25"].\
interpolate(method="time",limit=SMALL_GAP_THRESHOLD)

df_imputation.loc[small_gap_mask_o3, "o3"] = df_imputation["o3"].\
interpolate(method="time",limit=SMALL_GAP_THRESHOLD)



#### Medium imputation case (>6 and <=12 hours)
using KNN imputation and lag freatures

In [ ]:
from sklearn.impute import KNNImputer
import numpy as np

medium_gap_mask_pm25 = (df_merged["pm25_gap_length"] > SMALL_GAP_THRESHOLD) & \
                        (df_merged["pm25_gap_length"] <= MEDIUM_GAP_THRESHOLD)
medium_gap_mask_o3 = (df_merged["o3_gap_length"] > SMALL_GAP_THRESHOLD) & \
                        (df_merged["o3_gap_length"] <= MEDIUM_GAP_THRESHOLD)


#cyclic features
df_imputation["hour_sin"]=np.sin(2 * np.pi * df_imputation['hour']/24)
df_imputation["hour_cos"]=np.cos(2 * np.pi * df_imputation['hour']/24)
cols = ['pm25','o3','hour_sin','hour_cos']

imp = KNNImputer(n_neighbors=6)
imputed_knn= imp.fit_transform(df_imputation[cols])
df_all_imputed = pd.DataFrame(imputed_knn, columns=cols, index=df_imputation.index)
                               
                               



In [62]:
df_all_imputed

,pm25,o3,hour_sin,hour_cos
date,,,,
2017-03-03 05:00:00+05:45,106.1,0.002,0.965926,2.588190e-01
2017-03-03 06:00:00+05:45,134.5,0.002,1.000000,6.123234e-17
2017-03-03 07:00:00+05:45,154.7,0.002,0.965926,-2.588190e-01
2017-03-03 08:00:00+05:45,155.4,0.003,0.866025,-5.000000e-01
2017-03-03 09:00:00+05:45,178.7,0.005,0.707107,-7.071068e-01
...,...,...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039,-0.866025,5.000000e-01
2021-03-12 21:00:00+05:45,58.0,0.030,-0.707107,7.071068e-01
2021-03-12 22:00:00+05:45,60.0,0.030,-0.500000,8.660254e-01


In [ ]:
df_imputation.loc[medium_gap_mask_pm25,"pm25"]= df_all_imputed.loc[medium_gap_mask_pm25,"pm25"]
df_imputation.loc[medium_gap_mask_pm25,"was_imputed"]=1
df_imputation.loc[medium_gap_mask_o3,"o3"]= df_all_imputed.loc[medium_gap_mask_o3,"o3"]
df.imputation.loc[medium_gap_mask_o3,"was_imputed"]=1

pm25                  0
o3                    0
pm25_missing          0
o3_missing            0
pm25_gap_length       0
o3_gap_length         0
was_imputed        3647
hour                  0
dtype: int64

### Imputation of missing values

## Missing value in series check using gaps (not valid for this case experimentation)
larger gaps can be problematic ~ 20% of data is missing, setting the gap groups

Segmenting gaps to make sure they work as independent series for imputation

mapping the gap length dataframe back to the main dataframe

In [49]:
# df_merged['is_missing'] = df_merged.isna().any(axis=1).astype(int) #checking missing data
# df_merged['gap_group'] = (df_merged['is_missing'] != df_merged['is_missing'].shift()).cumsum() # summing the gaps using gap groups

# gap_lengths = ( # new dataframe with index of gap gruop
#     df_merged[df_merged['is_missing'] == 1]
#     .groupby('gap_group')
#     .size() #counting gap size with groupby of missing daatas
)

SyntaxError: unmatched ')' (1924040751.py, line 8)

In [ ]:


# df_merged["gap_length"] = (df_merged['gap_group'].map(gap_lengths).fillna(0).astype(int))

interpolation


In [ ]:
# df_merged["was_imputed"] = 0
# LONG_GAP = 24  # donot impute
# SHORT_GAP = 6  # impute
# cols = ['pm25', 'o3']

# small_gap_mask = (df_merged['is_missing'] == 1) & (df_merged['gap_length'] <= SHORT_GAP)

# if df_merged[]




In [ ]:
gap_lengths[gap_lengths > LONG_GAP].index

In [ ]:
# df_merged['segment_id'] = 0

# for g in gap_lengths[gap_lengths > LONG_GAP].index:
#     df_merged.loc[df_merged['gap_group'] >= g, 'segment_id'] += 1